# 🧱 Lab: Black-Box Run

**Module: Delivery: CLI, API, and Debugging Lab 2** | **Type: Wall Lab**

---

## Learning Objectives

By the end of this lab, you will be able to:

1. **Execute** a black-box verification pass — using only the project's documented commands, never reading the source code
2. **Identify** documentation bugs (wrong commands, missing steps, incorrect output) that prevent a newcomer from running the project
3. **Fix** every doc bug you find so the README becomes a reliable, reproducible runbook
4. **Apply** the CLI conventions (exit codes, stdout/stderr, argparse) learned earlier to verify correct behaviour from the outside

## Concepts Covered

| Concept | Gloss | From |
|---------|-------|------|
| Command-Line Interface | A CLI contract defines how a command is used and how it behaves: its arguments and flags, what `--help` promises, which exit codes mean success or failure, and what goes to stdout versus stderr. | mini-build-cli |
| Layered Configuration | Defaults live in code, environments override them at deploy time, and secrets never go in Git, so you can change behavior safely without editing source. | mini-build-cli |
| Interface Seam | A small typed contract between your app and an AI vendor. Your adapter returns your dataclass, never their SDK response object, so providers can be swapped without rewrites. | mini-define-ai-protocol |
| Debugging Decision Procedure | A repeatable debugging procedure: reproduce the failure, localise it to one pipeline box, shrink the input, then change one thing at a time until the cause is clear. | lab-runtime-failure-debugging |
| Filesystem Paths | Use Path objects, not strings, to build file locations that work across OSes. Prefer relative paths anchored to your project (not your shell), since relative paths depend on the current working directory. | lab-runtime-failure-debugging |
| Integration Tests | Exercise the shipped entry point, not just internal functions, so you catch real wiring issues across components, config, and dependencies. | mini-build-cli |
| Black-Box Verification | Use only documented commands and interfaces to run the system like a new user, every stumble is a documentation bug or missing contract. | This lab |
| Reproducibility | Someone else's machine reproduces your result from documented, pinned steps, starting from a fresh repo clone with no hidden setup. | lab-clone-and-run-check |

## Prerequisites

- **mini-build-cli**: Building CLIs with argparse, exit codes, stdout vs stderr, layered configuration, and a subprocess integration test of the shipped entry point
- **lab-runtime-failure-debugging**: Diagnosing runtime-only failures (env vars, paths, ports)

## What Is a Black-Box Run?

Imagine you've never seen this project before. You clone the repo, open the README, and follow the instructions **exactly as written**. You don't peek at the source. You don't guess what a command *should* be — you type what the docs *say*.

This is a **black-box verification pass** (sometimes called "The Stranger Test").

| Rule | Why |
|------|-----|
| Only run commands printed in the docs | If the docs don't say it, a newcomer won't know it |
| Don't read the source to figure out flags | The README is the contract — if it's wrong, the README is the bug |
| Record every failure | Each failure is a doc bug to fix |
| Fix the docs, not just the code | A working app with broken docs is still broken for the next person |

**Why this matters for reproducibility:** If someone can't clone your repo and get a working app by following the README alone, your project isn't reproducible — regardless of how clean your code is.

## Setup

In [1]:
import subprocess, sys, os, json, shutil
from pathlib import Path

from IPython.display import display, Markdown

def md(text):
    """Display text as rendered markdown."""
    display(Markdown(text))

def run(cmd, env=None, cwd=None):
    """Run a shell command and return result. Print stdout, stderr, exit code."""
    merged_env = {**os.environ, **(env or {})}
    result = subprocess.run(
        cmd, shell=True, capture_output=True, text=True,
        env=merged_env, cwd=cwd,
    )
    if result.stdout:
        print("[stdout]", result.stdout, end="")
    if result.stderr:
        print("[stderr]", result.stderr, end="")
    print(f"[exit code] {result.returncode}")
    return result

## Build the Project (with Planted Doc Bugs)

We'll create a small "LinkChecker" CLI project — a tool that takes a text file and counts how many URLs it contains. The **code works correctly**, but the **README has five documentation bugs** that will prevent a stranger from successfully running it.

Your job: play the stranger, follow the README, find every bug, and fix them.

In [2]:
# Create project structure
proj = Path("linkchecker")
proj.mkdir(exist_ok=True)
(proj / "samples").mkdir(exist_ok=True)
print("Project directory created.")

Project directory created.


In [3]:
%%writefile linkchecker/core.py
"""Core logic — find URLs in text."""
import re

URL_PATTERN = re.compile(r'https?://[^\s<>"\')]+', re.IGNORECASE)


def find_urls(text: str) -> dict:
    """Find all URLs in text. Returns dict with urls list and count."""
    urls = URL_PATTERN.findall(text)
    unique = sorted(set(urls))
    return {"urls": unique, "total": len(urls), "unique": len(unique)}

Writing linkchecker/core.py


In [4]:
%%writefile linkchecker/cli.py
"""CLI interface seam for LinkChecker."""
import argparse
import json
import os
import sys
from pathlib import Path

from core import find_urls


def main():
    parser = argparse.ArgumentParser(
        description="Find and count URLs in a text file."
    )
    parser.add_argument("file", help="Path to the text file to scan")
    parser.add_argument(
        "--json", action="store_true", dest="as_json",
        help="Output results as JSON",
    )
    parser.add_argument(
        "--unique-only", action="store_true",
        help="Only show unique URL count",
    )
    args = parser.parse_args()

    path = Path(args.file)
    if not path.exists():
        print(f"error: file not found: {path}", file=sys.stderr)
        sys.exit(1)

    text = path.read_text(encoding="utf-8")
    result = find_urls(text)

    if args.as_json:
        if args.unique_only:
            print(json.dumps({"unique": result["unique"], "urls": result["urls"]}))
        else:
            print(json.dumps(result))
    else:
        if args.unique_only:
            print(f"Unique URLs: {result['unique']}")
        else:
            print(f"Total URLs: {result['total']} ({result['unique']} unique)")
            for url in result["urls"]:
                print(f"  {url}")


if __name__ == "__main__":
    main()

Writing linkchecker/cli.py


In [5]:
%%writefile linkchecker/samples/example.txt
Check out https://docs.python.org for Python docs.
Also see https://github.com/python and https://docs.python.org again.
More info at http://example.com/resources.

Writing linkchecker/samples/example.txt


Now let's write the **buggy README**. This is the document a stranger would follow. It has **five doc bugs** planted in it — wrong commands, missing steps, and incorrect expected output.

In [6]:
%%writefile linkchecker/README.md
# LinkChecker CLI

A command-line tool that finds and counts URLs in text files.

## Installation

No dependencies required — uses only the Python standard library.

## Usage

### Basic usage

```bash
python linkcheck.py samples/example.txt
```

Expected output:
```
Total URLs: 4 (3 unique)
  http://example.com/resources
  https://docs.python.org
  https://github.com/python
```

### JSON output

```bash
python cli.py --output-json samples/example.txt
```

### Unique URLs only

```bash
python cli.py --unique samples/example.txt
```

Expected output:
```
Unique URLs: 3
```

### Run tests

```bash
pytest tests/
```

Writing linkchecker/README.md


The project is ready. Now let's **become the stranger**.

---

## The Black-Box Pass

Run every README command exactly as written. Every stumble is a doc bug: log it, then fix the README.

> 🧩 **Stranger / black-box run.** The stranger run from the video, now on a real README.

In [7]:
# Bug tracker — we'll append to this as we find issues
doc_bugs = []

In [8]:
readme = Path("linkchecker/README.md").read_text(encoding="utf-8")
md(readme)

# LinkChecker CLI

A command-line tool that finds and counts URLs in text files.

## Installation

No dependencies required — uses only the Python standard library.

## Usage

### Basic usage

```bash
python linkcheck.py samples/example.txt
```

Expected output:
```
Total URLs: 4 (3 unique)
  http://example.com/resources
  https://docs.python.org
  https://github.com/python
```

### JSON output

```bash
python cli.py --output-json samples/example.txt
```

### Unique URLs only

```bash
python cli.py --unique samples/example.txt
```

Expected output:
```
Unique URLs: 3
```

### Run tests

```bash
pytest tests/
```


### Command 1 — Basic Usage

The README says to run:
```bash
python linkcheck.py samples/example.txt
```

Let's try it exactly as written, from inside the project directory (since the README doesn't say otherwise).

In [9]:
# Run EXACTLY what the README says
result = run(
    f"{sys.executable} linkcheck.py samples/example.txt",
    cwd="linkchecker",
)

[stderr] C:\Users\molak\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe: can't open file 'C:\\Users\\molak\\MJ_Projects\\19_atomic_ai_eng_course\\1_ai_engineering_track\\output\\content\\labs\\lab-black-box-run\\linkchecker\\linkcheck.py': [Errno 2] No such file or directory
[exit code] 2


**❌ Failure!** The file is called `cli.py`, not `linkcheck.py`. This is Doc Bug #1.

In [10]:
doc_bugs.append({
    "id": 1,
    "section": "Basic usage",
    "documented_command": "python linkcheck.py samples/example.txt",
    "actual_error": "No such file: linkcheck.py (file is cli.py)",
    "fix": "Change 'linkcheck.py' to 'cli.py' in the Basic usage section",
})
print(f"Doc bug #{len(doc_bugs)} logged.")

Doc bug #1 logged.


Let's correct the command and try again to see if the rest of the basic usage section is accurate.

In [11]:
# Fix the filename and try again
result = run(
    f"{sys.executable} cli.py samples/example.txt",
    cwd="linkchecker",
)

[stdout] Total URLs: 4 (3 unique)
  http://example.com/resources.
  https://docs.python.org
  https://github.com/python
[exit code] 0


Basic usage works with the corrected filename ✅.

### Command 2 — JSON Output

The README says:
```bash
python cli.py --output-json samples/example.txt
```

In [12]:
# Run EXACTLY what the README says
result = run(
    f"{sys.executable} cli.py --output-json samples/example.txt",
    cwd="linkchecker",
)

[stderr] usage: cli.py [-h] [--json] [--unique-only] file
cli.py: error: unrecognized arguments: --output-json
[exit code] 2


**❌ Failure!** `argparse` doesn't recognize `--output-json`. Let's check what the CLI actually accepts by using `--help` — this is what a stranger would do when a flag doesn't work.

In [13]:
# A stranger's natural next step: check --help
run(f"{sys.executable} cli.py --help", cwd="linkchecker")

[stdout] usage: cli.py [-h] [--json] [--unique-only] file

Find and count URLs in a text file.

positional arguments:
  file           Path to the text file to scan

options:
  -h, --help     show this help message and exit
  --json         Output results as JSON
  --unique-only  Only show unique URL count
[exit code] 0


CompletedProcess(args='C:\\Users\\molak\\.virtualenvs\\text_llm_agent\\Scripts\\python.exe cli.py --help', returncode=0, stdout='usage: cli.py [-h] [--json] [--unique-only] file\n\nFind and count URLs in a text file.\n\npositional arguments:\n  file           Path to the text file to scan\n\noptions:\n  -h, --help     show this help message and exit\n  --json         Output results as JSON\n  --unique-only  Only show unique URL count\n', stderr='')

The help text shows `--json`, not `--output-json`. Doc Bug #2.

In [14]:
doc_bugs.append({
    "id": 2,
    "section": "JSON output",
    "documented_command": "python cli.py --output-json samples/example.txt",
    "actual_error": "Unrecognized argument --output-json (correct flag is --json)",
    "fix": "Change '--output-json' to '--json'",
})
print(f"Doc bug #{len(doc_bugs)} logged.")

Doc bug #2 logged.


In [15]:
# Verify with the correct flag
result = run(
    f"{sys.executable} cli.py --json samples/example.txt",
    cwd="linkchecker",
)

[stdout] {"urls": ["http://example.com/resources.", "https://docs.python.org", "https://github.com/python"], "total": 4, "unique": 3}
[exit code] 0


JSON output works with the correct flag ✅. Next: Unique URLs only.

### Command 3 — Unique URLs Only

The README says:
```bash
python cli.py --unique samples/example.txt
```

In [16]:
# Run EXACTLY what the README says
result = run(
    f"{sys.executable} cli.py --unique samples/example.txt",
    cwd="linkchecker",
)

[stdout] Unique URLs: 3
[exit code] 0


**⚠️ It ran, but only by accident.** `--unique` is not a real flag: the `--help` output above lists `--unique-only`. `argparse` accepts any unambiguous prefix of a long option, so `--unique` quietly expanded to `--unique-only`.

The README is relying on an abbreviation, not the documented interface. Add another flag that starts with `--unique`, or set `allow_abbrev=False`, and this documented command breaks. Doc Bug #3.

In [17]:
doc_bugs.append({
    "id": 3,
    "section": "Unique URLs only",
    "documented_command": "python cli.py --unique samples/example.txt",
    "actual_error": "Runs only via argparse prefix matching; --unique is not in --help (real flag is --unique-only)",
    "fix": "Change '--unique' to '--unique-only'",
})
print(f"Doc bug #{len(doc_bugs)} logged.")

Doc bug #3 logged.


In [18]:
# Verify with the correct flag
result = run(
    f"{sys.executable} cli.py --unique-only samples/example.txt",
    cwd="linkchecker",
)

[stdout] Unique URLs: 3
[exit code] 0


Unique-only output matches ✅. Next: tests.

### Command 4 — Run Tests

The README says:
```bash
pytest tests/
```

In [19]:
# Run EXACTLY what the README says
result = run(
    f"{sys.executable} -m pytest tests/",
    cwd="linkchecker",
)

[stdout] ============================= test session starts =============================
platform win32 -- Python 3.12.10, pytest-9.1.1, pluggy-1.6.0
rootdir: C:\Users\molak\MJ_Projects\19_atomic_ai_eng_course\1_ai_engineering_track\output\content\labs\lab-black-box-run\linkchecker
plugins: anyio-4.15.1, langsmith-0.14.1, asyncio-1.4.0
asyncio: mode=Mode.STRICT, debug=False, asyncio_default_fixture_loop_scope=None, asyncio_default_test_loop_scope=function
collected 0 items

============================ no tests ran in 0.01s ============================
[stderr] ERROR: file or directory not found: tests/

[exit code] 4


**❌ Failure!** There is no `tests/` directory. The README documents a test command, but no tests exist in the project. Doc Bug #4.

This is a common doc bug: the README was written optimistically ("we'll add tests later") but the section was never removed or updated.

In [20]:
doc_bugs.append({
    "id": 4,
    "section": "Run tests",
    "documented_command": "pytest tests/",
    "actual_error": "No 'tests/' directory exists in the project",
    "fix": "Remove the 'Run tests' section (or create the tests)",
})
print(f"Doc bug #{len(doc_bugs)} logged.")

Doc bug #4 logged.


### Command 5 — Working Directory

The README doesn't say where to run commands. Relative paths like `samples/example.txt` only work from inside the project directory, so this is Doc Bug #5.

In [21]:
doc_bugs.append({
    "id": 5,
    "section": "Usage (top)",
    "documented_command": "(no cd instruction before relative-path commands)",
    "actual_error": "Relative paths like 'samples/example.txt' fail unless user is in the project dir",
    "fix": "Add 'cd linkchecker' instruction before the usage examples",
})
print(f"Doc bug #{len(doc_bugs)} logged.")

Doc bug #5 logged.


In [22]:
# Review the bug log before fixing the README
for b in doc_bugs:
    print(f"#{b['id']} ({b['section']}): {b['fix']}")

#1 (Basic usage): Change 'linkcheck.py' to 'cli.py' in the Basic usage section
#2 (JSON output): Change '--output-json' to '--json'
#3 (Unique URLs only): Change '--unique' to '--unique-only'
#4 (Run tests): Remove the 'Run tests' section (or create the tests)
#5 (Usage (top)): Add 'cd linkchecker' instruction before the usage examples


---

## Fix the README

Now we apply all five fixes and write a corrected README. The stranger test principle: **fix the docs, not just the code**.

In [23]:
%%writefile linkchecker/README.md
# LinkChecker CLI

A command-line tool that finds and counts URLs in text files.

## Installation

No dependencies required — uses only the Python standard library.

## Usage

First, navigate to the project directory:

```bash
cd linkchecker
```

### Basic usage

```bash
python cli.py samples/example.txt
```

Expected output:
```
Total URLs: 4 (3 unique)
  http://example.com/resources
  https://docs.python.org
  https://github.com/python
```

### JSON output

```bash
python cli.py --json samples/example.txt
```

### Unique URLs only

```bash
python cli.py --unique-only samples/example.txt
```

Expected output:
```
Unique URLs: 3
```

Overwriting linkchecker/README.md


---

## Verify the Fixed README

The final step of a black-box pass: **re-run every documented command** to confirm the README is now accurate. This is the reproducibility check.

In [24]:
# Re-run every documented command from the fixed README
r1 = run(f"{sys.executable} cli.py samples/example.txt", cwd="linkchecker")
r2 = run(f"{sys.executable} cli.py --json samples/example.txt", cwd="linkchecker")
r3 = run(f"{sys.executable} cli.py --unique-only samples/example.txt", cwd="linkchecker")

# Minimal checks: every command succeeds, and the JSON output parses
assert r1.returncode == 0 and r2.returncode == 0 and r3.returncode == 0
json.loads(r2.stdout)

print("✅ All documented commands work correctly!")

[stdout] Total URLs: 4 (3 unique)
  http://example.com/resources.
  https://docs.python.org
  https://github.com/python
[exit code] 0


[stdout] {"urls": ["http://example.com/resources.", "https://docs.python.org", "https://github.com/python"], "total": 4, "unique": 3}
[exit code] 0


[stdout] Unique URLs: 3
[exit code] 0
✅ All documented commands work correctly!


In [25]:
# A reusable black-box verification checklist for any project
checklist = """
| # | Check | Question to Ask |
|---|-------|-----------------|
| 1 | **Every command** | Does every code block in the README run exactly as written? |
| 2 | **Flag names** | Do all documented CLI flags exist? (`--help` is your friend) |
| 3 | **Working directory / paths** | Does the README say where to `cd`, and do all referenced files exist? |
| 4 | **Prerequisites** | Are all required env vars, tools, and dependencies listed? |
| 5 | **Expected output** | Does the actual output match what the README claims? |
"""
md(checklist)


| # | Check | Question to Ask |
|---|-------|-----------------|
| 1 | **Every command** | Does every code block in the README run exactly as written? |
| 2 | **Flag names** | Do all documented CLI flags exist? (`--help` is your friend) |
| 3 | **Working directory / paths** | Does the README say where to `cd`, and do all referenced files exist? |
| 4 | **Prerequisites** | Are all required env vars, tools, and dependencies listed? |
| 5 | **Expected output** | Does the actual output match what the README claims? |


In [26]:
shutil.rmtree("linkchecker", ignore_errors=True)
print("Cleaned up.")

Cleaned up.


## 🎯 Summary

### What You Built

You performed a complete black-box verification pass on a CLI project — following only the documented commands, discovering five doc bugs, fixing every one, and verifying the corrected README end-to-end.

### Key Takeaways

1. **Black-box verification** — run every documented command exactly as written, without reading source code; if a command fails, the documentation is the bug
2. **Doc bugs are real bugs** — a wrong filename, incorrect flag, or missing `cd` instruction will stop every newcomer; broken docs break reproducibility just as much as broken code
3. **`--help` is the stranger's first tool** — when a documented flag doesn't work, `--help` reveals the actual interface without reading source
4. **Fix the docs, then re-verify** — after correcting all bugs, re-run every command to confirm the README is now a reliable, reproducible runbook
5. **Reproducibility requires a complete path** — from `cd` into the directory to the final output, every step must be documented or the project isn't reproducible for anyone but the author